# Weighted Linear Least Squares

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/themintlab/ExecutableEngineering/blob/main/chapters/interpolation_and_curve_fitting/curve_fitting/weighted_linear_least_squares.ipynb)


In [ ]:
import numpy as np
import plotly.graph_objects as go

try:
    import executable_engineering as exe
except ImportError:
    %pip install -q executable_engineering
    import executable_engineering as exe


## Recall: The Pseudoinverse

In the Linear Systems chapter, we introduced the concept of the **pseudoinverse** as a method to find the least squares solution to an overdetermined system $\mathbf{A}\mathbf{x} \approx \mathbf{b}$. 

By minimizing the residual norm $\|\mathbf{A}\mathbf{x} - \mathbf{b}\|$, we obtain the best fit solution $\mathbf{x} = \mathbf{A}^+ \mathbf{b}$. We will now build upon this concept for curve fitting.

## The Problem with Unweighted Residuals

As with most approximate methods, the moment we start to move away from an *exact solution* subtle effects start to show up! 

Notice we are minimizing the *residuals* but there is a subtle problem with the problem definition above:

$$\begin{aligned}
20 c + 50 t &= 700  \\
c+t  &= 20 \\
60 c + 20 t &= 700
\end{aligned} $$

The coefficients of the second equation are about an order of magnitude lower than the others. Of course this system is equivalent to:

$$\begin{aligned}
20 c + 50 t &= 700  \\
10c+10t  &= 200 \\
60 c + 20 t &= 700
\end{aligned} $$

or even

$$\begin{aligned}
0.2 c + 0.5 t &= 7  \\
c+t  &= 20 \\
0.6 c + 0.2 t &= 7
\end{aligned} $$

What does this remind you of?

## Introducing Weights

Jacobi (diagonal) preconditioning!

$$P^{-1} = \begin{bmatrix} W_{1} & 0 & 0 \\ 0 & W_{2} & 0 \\ 0 & 0 & W_{3} \end{bmatrix}$$

If we define $r_1, r_2, r_3$:

$$\begin{aligned}
20 c + 50 t -700 &= r_1 \\
c+t -20 &= r_2 \\
60 c + 20 t -700 &= r_3
\end{aligned} $$

We say the residuals can be *weighted*, i.e.: the least squares problem becomes,

$$ \text{Min}_{\mathbf{x}} \ \sum W_i^2 r_i^2 $$

Let's code it and see how applying this diagonal matrix shifts the least squares solution!

In [ ]:
A = np.array([[20, 50], [1, 1], [60, 20]])
b = np.array([700, 20, 700])

#~~ Question: What's the preconditioner? How do we apply it?
Pi = np.diag([1, 7, 1])
print("Preconditioner / Weights Matrix:\n", Pi)
A_weighted = Pi @ A
b_weighted = Pi @ b
###

x_lsq, _, _, _ = np.linalg.lstsq(A_weighted, b_weighted, rcond=None)
print("\nLeast Squares Solution (c, t):", x_lsq)

# Prepare data for plotting
x_line = np.linspace(5, 15, 100)
y1 = (700 - 20*x_line) / 50
y2 = 20 - x_line
y3 = (700 - 60*x_line) / 20

# Plot the lines and the solution
fig = go.Figure()
fig.add_trace(go.Scatter(x=x_line, y=y1, mode='lines', name='20c + 50t = 700'))
fig.add_trace(go.Scatter(x=x_line, y=y2, mode='lines', name='c + t = 20'))
fig.add_trace(go.Scatter(x=x_line, y=y3, mode='lines', name='60c + 20t = 700'))
fig.add_trace(go.Scatter(x=[x_lsq[0]], y=[x_lsq[1]], mode='markers', 
                         marker=dict(size=12, color='red', symbol='star'), 
                         name='Weighted Least Squares Solution'))

fig.update_layout(title='Weighted Least Squares Solution for Inconsistent System', 
                  xaxis_title='c', yaxis_title='t')
fig.show()


## Measurement Uncertainty as Weights

Weights are an excellent way to introduce measurement uncertainty into your fit!

If you have multiple measurements, $y_i^j$, at a single point, $x_i$, then you can find the mean,

$$
\tilde{y}_i = \text{mean}_j (y_i^j)   
$$

and its variance,

$$
\sigma_i^2 = \text{mean}\left([y_i^j - \tilde{y}_i]^2 \right)
$$ 

and weight by the inverse of the variance, $W_i = \frac{1}{\sigma_i^2}$.

Most package implementations of weighted least squares regression have an option to input the weights, but exact usage varies. Consult the documentation to determine how to use it in practice.